# Progressive Growing GAN（PGGAN）

---
## 目的
Progressive Growing GAN (PGGAN) [1]を構築し，低解像度から高解像度へ段階的にネットワークを成長させながら学習する仕組みを理解する．本ノートブックでは，`dcgan.ipynb`, `wgan_gp.ipynb`で扱った$32\times32$のMNIST画像を最終的な目標解像度とし，$4\times4 \to 8\times8 \to 16\times16 \to 32\times32$の4段階でネットワークを成長させながら学習する．

[1] Tero Karras, Timo Aila, Samuli Laine, Jaakko Lehtinen, "Progressive Growing of GANs for Improved Quality, Stability, and Variation," ICLR, 2018.

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import os
import zipfile
from time import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Progressive Growing GANとは
これまでのGAN系ノートブックは，最初から目標解像度（例．$32\times32$）のGenerator・Discriminatorを学習していました．高解像度な画像を生成しようとすると，Generator・Discriminatorのネットワークが大きくなり，学習が不安定になりやすいという問題があります．

PGGAN[1]は，**最初は$4\times4$という非常に低い解像度だけを学習し，学習が安定してきたら層を追加して解像度を2倍にする**，という手順を繰り返すことで，高解像度画像の生成を段階的に学習します．低解像度の段階でまず大まかな構図（数字であれば大まかな形）を学習し，解像度を上げるごとに細部（数字のエッジなど）を学習していくイメージです．

### Fade-in（滑らかな移行）
新しい解像度の層を突然追加すると，それまで安定して学習していたネットワークに急激な変化（ショック）を与えてしまいます．そこでPGGANでは，新しい層を追加する際に，**Fade-in**という滑らかな移行の仕組みを導入します．新しい解像度に切り替わってから一定期間は，「1つ前の解像度の出力をアップサンプリングしたもの」と「新しい解像度の層の出力」を，重み$\alpha$（$0\to1$に徐々に増加）で線形に混合した画像を出力します．$\alpha=0$では旧layerの出力そのもの，$\alpha=1$では新しいlayerの出力そのものになります．

## Equalized Learning Rate
PGGANでは，各層の重みを標準正規分布$N(0,1)$で初期化したうえで，順伝播時に，その層のfan-in（入力ユニット数）に応じたHe初期化相当のスケール定数を毎回乗算する，**Equalized Learning Rate**という手法を用います．

通常のネットワークでは，層ごとにfan-inが異なるとパラメータの実効的な更新量にも差が生じ，学習が不安定になることがあります．Equalized Learning Rateは，全ての層のパラメータをほぼ同じ実効的な学習率で更新できるようにすることで，学習を安定させます．`nn.Conv2d`や`nn.Linear`をそのまま使う代わりに，以下の`EqualizedConv2d`, `EqualizedLinear`を使用します．

In [ ]:
class EqualizedConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_ch))
        fan_in = in_ch * kernel_size * kernel_size
        self.scale = (2 / fan_in) ** 0.5  # He初期化相当のスケール
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        return F.conv2d(x, self.weight * self.scale, self.bias, stride=self.stride, padding=self.padding)


class EqualizedLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f))
        self.bias = nn.Parameter(torch.zeros(out_f))
        self.scale = (2 / in_f) ** 0.5

    def forward(self, x):
        return F.linear(x, self.weight * self.scale, self.bias)

## Pixel Normalization
`dcgan.ipynb`などのGeneratorはBatch Normalizationを使用していましたが，Batch Normalizationはミニバッチ内の他のサンプルの統計量に依存するため，PGGANのように学習の途中でネットワーク構造（解像度）が変化する状況とは相性がよくありません．

そこでPGGANのGeneratorでは，**Pixel Normalization**という正規化を使用します．これは，各画素位置（$H\times W$の各座標）ごとに，チャネル方向の特徴ベクトルをノルム1に正規化する処理で，他のサンプルの統計量に依存しないという特徴があります．

In [ ]:
class PixelNorm(nn.Module):
    def forward(self, x):
        return x / torch.sqrt(torch.mean(x ** 2, dim=1, keepdim=True) + 1e-8)

## Generator
Generatorは，解像度ごとの処理単位である`GBlock`を積み重ねて構築します．各`GBlock`は，（最初のブロックを除いて）入力を2倍にアップサンプリングしたのち，畳み込み・Pixel Normalization・Leaky ReLUを2回繰り返します．最初のブロックだけは，$1\times1$の潜在変数を$4\times4$の特徴マップへ拡張する特殊な処理を行います．

各解像度の特徴マップは，`to_image`という$1\times1$畳み込み層によって画像（チャネル数1のグレースケール画像）へ変換されます．`forward`では，現在の成長段階`stage`（0: $4\times4$のみ, 1: $8\times8$まで, 2: $16\times16$まで, 3: $32\times32$まで）と，Fade-inの係数`alpha`を受け取り，該当する解像度までの処理を行います．

In [ ]:
class GBlock(nn.Module):
    def __init__(self, in_ch, out_ch, first_block=False):
        super().__init__()
        self.first_block = first_block
        if first_block:
            self.conv1 = EqualizedConv2d(in_ch, out_ch, kernel_size=4, stride=1, padding=3)  # 1x1 -> 4x4
        else:
            self.conv1 = EqualizedConv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1)
        self.conv2 = EqualizedConv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1)
        self.pixelnorm = PixelNorm()
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        if not self.first_block:
            x = F.interpolate(x, scale_factor=2, mode='nearest')  # 解像度を2倍にする
        x = self.act(self.pixelnorm(self.conv1(x)))
        x = self.act(self.pixelnorm(self.conv2(x)))
        return x


class Generator(nn.Module):
    def __init__(self, latent_dim=100, channels=(256, 256, 128, 64), out_ch=1):
        super().__init__()
        # channels[i]: 段階i（4x4, 8x8, 16x16, 32x32）における特徴マップのチャネル数
        self.blocks = nn.ModuleList([GBlock(latent_dim, channels[0], first_block=True)])
        self.to_image = nn.ModuleList([EqualizedConv2d(channels[0], out_ch, kernel_size=1)])
        for i in range(1, len(channels)):
            self.blocks.append(GBlock(channels[i - 1], channels[i]))
            self.to_image.append(EqualizedConv2d(channels[i], out_ch, kernel_size=1))

    def forward(self, z, stage, alpha=1.0):
        x = z
        feats = [z]
        for i in range(stage + 1):
            x = self.blocks[i](x)
            feats.append(x)

        out_new = self.to_image[stage](feats[stage + 1])
        if stage == 0 or alpha >= 1.0:
            return torch.sigmoid(out_new)

        # Fade-in：1つ前の解像度の出力をアップサンプリングしたものと，新しい解像度の出力を混合する
        out_old = self.to_image[stage - 1](feats[stage])
        out_old_up = F.interpolate(out_old, scale_factor=2, mode='nearest')
        blended = alpha * out_new + (1 - alpha) * out_old_up
        return torch.sigmoid(blended)

## Minibatch Standard Deviation
GANでは，Generatorが多様性のない（似たような画像ばかりを生成する）状態に陥る**モード崩壊**が問題になることがあります．PGGANのDiscriminatorは，ミニバッチ内の各特徴マップの標準偏差を計算し，その平均値を1チャンネルの特徴マップとして追加する**Minibatch Standard Deviation**という層を，最終ブロックの直前に挿入します．

これにより，Discriminatorはミニバッチ全体の多様性の情報を利用できるようになり，生成画像の多様性が乏しい（＝標準偏差が小さい）場合にそれを見抜きやすくなります．結果として，Generatorは多様な画像を生成するように学習されます．

In [ ]:
class MinibatchStdDev(nn.Module):
    def forward(self, x):
        std = torch.std(x, dim=0, unbiased=False).mean()  # ミニバッチ内の各特徴の標準偏差の平均値
        std_map = std.expand(x.size(0), 1, x.size(2), x.size(3))
        return torch.cat([x, std_map], dim=1)  # 追加の1チャンネルとして連結

## Discriminator
Discriminatorは，Generatorと対称な構造をしています．各解像度の画像は，`from_image`という$1\times1$畳み込み層によって特徴マップへ変換されたのち，`DBlock`（畳み込み2回＋ダウンサンプリング）を通って徐々に低解像度化されます．最終ブロック（$4\times4\to1\times1$）の直前にのみ，Minibatch Standard Devを適用します．

`forward`では，Generatorと同様に`stage`・`alpha`を受け取り，現在の解像度の画像を，対応する`from_image`から入力してDiscriminatorへ通します．Fade-in中は，現在の解像度の特徴と，1つダウンサンプリングした画像を1つ前の解像度の`from_image`に通した特徴を，`alpha`で混合します．

In [ ]:
class DBlock(nn.Module):
    def __init__(self, in_ch, out_ch, last_block=False):
        super().__init__()
        self.last_block = last_block
        if last_block:
            self.minibatch_stddev = MinibatchStdDev()
            self.conv1 = EqualizedConv2d(in_ch + 1, out_ch, kernel_size=3, stride=1, padding=1)
            self.conv2 = EqualizedConv2d(out_ch, out_ch, kernel_size=4, stride=1, padding=0)  # 4x4 -> 1x1
        else:
            self.minibatch_stddev = None
            self.conv1 = EqualizedConv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1)
            self.conv2 = EqualizedConv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        if self.minibatch_stddev is not None:
            x = self.minibatch_stddev(x)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        if not self.last_block:
            x = F.avg_pool2d(x, 2)  # 解像度を半分にする
        return x


class Discriminator(nn.Module):
    def __init__(self, channels=(256, 256, 128, 64), in_ch=1):
        super().__init__()
        # channelsはGeneratorと同じ並び（0:4x4, 1:8x8, ...）で定義するが，
        # 実際の処理は高解像度側（stage）から低解像度側（0）へ向かって行う
        n = len(channels)
        self.blocks = nn.ModuleList()
        self.from_image = nn.ModuleList()
        for i in range(n):
            out_c = channels[i - 1] if i > 0 else channels[0]
            self.blocks.append(DBlock(channels[i], out_c, last_block=(i == 0)))
            self.from_image.append(EqualizedConv2d(in_ch, channels[i], kernel_size=1))
        self.final_linear = EqualizedLinear(channels[0], 1)

    def forward(self, img, stage, alpha=1.0):
        x = self.from_image[stage](img)
        x = self.blocks[stage](x)

        if stage > 0 and alpha < 1.0:  # Fade-in
            img_down = F.avg_pool2d(img, 2)
            x_old = self.from_image[stage - 1](img_down)
            x = alpha * x + (1 - alpha) * x_old

        for i in range(stage - 1, -1, -1):
            x = self.blocks[i](x)

        x = x.view(x.size(0), -1)
        return self.final_linear(x).view(-1)

## ネットワークの作成
Generator・Discriminatorを作成します．損失関数には，Wasserstein距離を使用します．1-Lipschitz制約を満たすための手法としては，`wgan_gp.ipynb`で扱ったGradient Penaltyの方が高品質な生成画像が得られやすいことが知られていますが，本ノートブックでは実装を簡略化するため，元のWGAN論文のWeight Clippingを使用します．そのため，最適化手法もWeight Clippingと相性の良いRMSpropを使用します（Gradient Penaltyとモーメンタムを用いる最適化手法の組み合わせについては`wgan_gp.ipynb`を参照してください）．

In [ ]:
channels = (256, 256, 128, 64)  # 4x4, 8x8, 16x16, 32x32 の各段階のチャネル数
resolutions = (4, 8, 16, 32)
latent_dim = 100

G = Generator(latent_dim=latent_dim, channels=channels, out_ch=1).to(device)
D = Discriminator(channels=channels, in_ch=1).to(device)

opt_g = torch.optim.RMSprop(G.parameters(), lr=5e-5)
opt_d = torch.optim.RMSprop(D.parameters(), lr=5e-5)

num_params = sum(p.numel() for p in G.parameters()) + sum(p.numel() for p in D.parameters())
print('パラメータ数（G+D）:', num_params)

## 学習
$4\times4 \to 8\times8 \to 16\times16 \to 32\times32$の順に，解像度を切り替えながら学習します．各解像度では，MNISTデータセットをその解像度にリサイズして使用します．$4\times4$の最初の段階を除き，各解像度の学習は，`fade_epochs`エポックかけて$\alpha$を$0\to1$に増加させるFade-in期間と，その後`stabilize_epochs`エポック$\alpha=1$で学習を安定させる期間の2段階に分けます．Critic（Discriminator）の学習は`wgan_gp.ipynb`と同様，`n_critic`回に1回Generatorを更新します．

※ Weight Clippingを適用する際の注意点として，`EqualizedConv2d`/`EqualizedLinear`が実際に順伝播で使う重みは「保持しているパラメータ×`scale`」であるため，保持しているパラメータをそのまま`clip_value`で切ってしまうと，実効的な重みの範囲が`scale`倍された非常に小さい値になってしまいます．そこで，各層の`scale`で割った範囲でパラメータをclipすることで，実効的な重みが`clip_value`の範囲に収まるようにしています．

In [ ]:
n_critic = 5
clip_value = 0.01
fade_epochs = 5
stabilize_epochs = 5
batch_size = 64

start = time()
for stage, res in enumerate(resolutions):
    transform = transforms.Compose([transforms.Resize((res, res)), transforms.ToTensor()])
    mnist_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
    train_loader = DataLoader(mnist_data, batch_size=batch_size, shuffle=True)

    epochs_this_stage = stabilize_epochs if stage == 0 else fade_epochs + stabilize_epochs
    for epoch in range(1, epochs_this_stage + 1):
        alpha = 1.0 if stage == 0 else min(1.0, epoch / fade_epochs)

        G.train()
        D.train()
        critic_count = 0
        for idx, (real_x, _) in enumerate(train_loader):
            real_x = real_x.to(device)
            batch = real_x.size(0)

            z = torch.randn(batch, latent_dim, 1, 1).to(device)
            fake_x = G(z, stage, alpha)
            d_loss = -(D(real_x, stage, alpha).mean() - D(fake_x.detach(), stage, alpha).mean())

            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()
            # Equalized Learning Rateレイヤーは実効的な重みが (保持している重み × scale) であるため，
            # 保持している重みをそのままclip_valueで切ると，実効的な重みの範囲が意図より大幅に小さくなってしまう．
            # そのため，各層のscaleで割った範囲でclipすることで，実効的な重みをclip_valueの範囲に収める．
            for m in D.modules():
                if isinstance(m, (EqualizedConv2d, EqualizedLinear)):
                    m.weight.data.clamp_(-clip_value / m.scale, clip_value / m.scale)
                    m.bias.data.clamp_(-clip_value, clip_value)

            critic_count += 1
            if critic_count % n_critic == 0:
                z = torch.randn(batch, latent_dim, 1, 1).to(device)
                fake_x = G(z, stage, alpha)
                g_loss = -D(fake_x, stage, alpha).mean()

                opt_g.zero_grad()
                g_loss.backward()
                opt_g.step()

        print(f'stage={stage} (res={res}x{res}), epoch: {epoch}/{epochs_this_stage}, alpha: {alpha:.2f}, '
              f'D loss: {-d_loss.item():.4f}, G loss: {g_loss.item():.4f}, elapsed_time: {time() - start:.4f}')

    # 1 stageごとにモデルを保存
    torch.save(G.state_dict(), "gen_stage_%d.pt" % stage)
    torch.save(D.state_dict(), "dis_stage_%d.pt" % stage)

## 学習済みモデルのダウンロード

PGGANの学習が演習時間中に終わらない場合は，下記のプログラムを実行して学習済みモデルをダウンロードし，結果を確認してください．

In [ ]:
model_root = './pggan_model'
if not os.path.isdir(model_root):
    gdown.download(id='1Xv3D8LQ54o5LPjVdwDTLehiAQYIPaDwU', output='pggan_model.zip', quiet=False)
    with zipfile.ZipFile('pggan_model.zip') as f:
        f.extractall('./')

G.load_state_dict(torch.load('./pggan_model/gen_stage_3.pt', map_location=device))
D.load_state_dict(torch.load('./pggan_model/dis_stage_3.pt', map_location=device))

## 学習したGeneratorによる画像生成
最終段階（$32\times32$）まで学習したGeneratorを用いて画像を生成します．

In [ ]:
num_generate = 100
z = torch.randn(num_generate, latent_dim, 1, 1).to(device)

G.eval()
with torch.no_grad():
    test_img = G(z, stage=len(resolutions) - 1, alpha=1.0)

test_img = test_img.view(num_generate, 32, 32).cpu().numpy()

fig = plt.figure(figsize=(10, 10))
for i, im in enumerate(test_img):
    ax = fig.add_subplot(10, 10, i + 1, xticks=[], yticks=[])
    ax.imshow(im, cmap='gray')
plt.show()

## 各解像度段階の生成結果の比較
学習済みのGeneratorは，`blocks[0]`から`blocks[3]`までの全ての解像度段階のパラメータを保持しています．同じ潜在変数`z`を用いて，`stage`引数を変えながら生成することで，同一のGeneratorが内部的に保持している各解像度の生成結果を比較できます．

In [ ]:
z = torch.randn(8, latent_dim, 1, 1).to(device)

G.eval()
fig, axes = plt.subplots(len(resolutions), 8, figsize=(1.4 * 8, 1.4 * len(resolutions)))
with torch.no_grad():
    for stage, res in enumerate(resolutions):
        imgs = G(z, stage=stage, alpha=1.0).view(-1, res, res).cpu().numpy()
        for i in range(8):
            axes[stage, i].imshow(imgs[i], cmap='gray')
            axes[stage, i].axis('off')
        axes[stage, 0].set_ylabel(f'{res}x{res}', rotation=0, labelpad=30)

plt.tight_layout()
plt.show()

## 課題

1. `fade_epochs`, `stabilize_epochs`を変更して学習し，各解像度への移行のスムーズさや最終的な生成画像の質がどのように変化するか確認してください．
2. Minibatch Standard Devの層を取り除いて学習し，生成される画像の多様性（例えば0〜9の数字がバランスよく生成されるか）にどのような変化が現れるか確認してください．
3. `channels`を変更（各段階のチャネル数を増減）して学習し，パラメータ数と生成画像の質のトレードオフを確認してください．